In [6]:
import numpy as np
import subprocess
import optuna
import matplotlib.pyplot as plt

In [4]:
control_indices = [1, 2, 3, 4, 6, 7]

params = {
        idx: 0
        for idx in control_indices
    }

cmd = ['./kmc_project']
for idx, val in params.items():
    cmd.append(f"--c_v={idx}={val}")

cmd.append(f"--input_idx={0}")
cmd.append(f"--output_idx={5}")
print(cmd)

['./kmc_project', '--c_v=1=0', '--c_v=2=0', '--c_v=3=0', '--c_v=4=0', '--c_v=6=0', '--c_v=7=0', '--input_idx=0', '--output_idx=5']


In [ ]:
def target_function(x: np.array) -> np.array:

    target = x**3

    return target

def pearson_correlation_coefficient(x: np.array, y: np.array) -> float:

    N = x.shape[0]

    x_bar = np.mean(x)
    y_bar = np.mean(y)
    std_x = np.std(x, ddof=1)
    std_y = np.std(y, ddof=1)

    x_c = (x - x_bar) / std_x
    y_c = (y - y_bar) / std_y

    pcc = np.sum(x_c*y_c) / (N - 1)

    return pcc   

def objective(trial):

    V_MIN = -1.5
    V_MAX = 1.5

    control_indices = [1, 2, 3, 4, 6, 7]
    input_index = 5
    output_index = 0

    params = {
        idx: trial.suggest_float(f"c_{idx}", V_MIN, V_MAX)
        for idx in control_indices
    }

    cmd = ['./kmc_project']
    for idx, val in control_indices.items():
        cmd.append(f"--c_v={idx}={val}")
    cmd.append(f"--input_idx={input_index[0]}")
    cmd.append(f"--output_idx={output_index[0]}")
        
    proc = subprocess.run(cmd, capture_output=True, text=True)

    file_name = f"../trials/data_point{trial.number}.npz"
    curve = np.load(file=file_name)

    score = pearson_correlation_coefficient(curve)

    return score